# PromptTemplate — 프롬프트를 재사용 가능한 틀로 만들기

Ch01에서 LLM 호출을 배웠다면, 이제 프롬프트를 체계적으로 관리하는 법을 배워요.

매번 프롬프트를 하드코딩하는 대신, 변수 슬롯이 있는 **템플릿(template)**을 만들어두면 같은 구조의 프롬프트를 여러 상황에서 재사용할 수 있어요.

## 학습 목표

- `PromptTemplate`으로 변수를 포함한 재사용 가능한 프롬프트를 만들 수 있어요
- `ChatPromptTemplate`으로 System / Human / AI 역할을 구분한 대화형 프롬프트를 구성할 수 있어요
- `partial()`과 동적 함수 변수로 프롬프트를 더 유연하게 다룰 수 있어요

## 사전 지식

- Ch01의 `ChatOpenAI` 기본 호출 방법
- LCEL 파이프라인 (`|` 연산자) 기초

---

> **PromptTemplate 동작 흐름** — 아래 다이어그램은 변수가 어떻게 최종 프롬프트로 조립되어 LLM에 전달되는지 보여줘요.

```mermaid
flowchart LR
    V1["변수 role<br/>'파이썬 전문가'"]:::input
    V2["변수 topic<br/>'데코레이터'"]:::input
    T["PromptTemplate<br/>템플릿 문자열<br/>'당신은 {role}입니다.<br/>{topic}에 대해 설명해주세요.'"]:::process
    P["완성된 프롬프트<br/>'당신은 파이썬 전문가입니다.<br/>데코레이터에 대해 설명해주세요.'"]:::output
    LLM["LLM<br/>ChatOpenAI"]:::external
    R["응답"]:::output

    V1 --> T
    V2 --> T
    T -->|"format() / invoke()"| P
    P --> LLM
    LLM --> R

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

## 1. PromptTemplate

`PromptTemplate`은 문자열 기반의 간단한 프롬프트를 생성하는 템플릿입니다.

### 주요 특징:
- 중괄호 `{}`를 사용하여 변수 정의
- `from_template()` 메서드로 간편하게 생성
- `format()` 또는 `invoke()`로 변수에 값을 채워 최종 프롬프트 생성


In [34]:
# 필수 라이브러리 import
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

# 모델 초기화
# temperature=0: 일관된 출력을 위해 설정
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

### 1.1 기본 사용법


In [25]:
# 1단계: 템플릿 정의 - 중괄호 {}로 변수 지정
prompt = PromptTemplate.from_template(
  "당신은 {role}입니다. {topic}에 대해 {length} 단어 이내로 설명해주세요"
)

res = prompt.format(role="100년차 개발자", topic="자바스크립트", length="100")
print(f' ==> [Line 6]: \033[38;2;140;180;8m[res]\033[0m({type(res).__name__}) = \033[38;2;62;176;189m{res}\033[0m')




 ==> [Line 6]: [res](str) = Human: 당신은 100년차 개발자입니다. 자바스크립트에 대해 100 단어 이내로 설명해주세요


### 1.2 체인과 함께 사용하기

`PromptTemplate`을 LLM과 연결하여 체인을 구성할 수 있습니다.


In [26]:
# 1단계: 템플릿 정의
prompt = PromptTemplate.from_template(
  "당신은 {role}입니다. {topic}에 대해 {length} 단어 이내로 설명해주세요"
)

# 2단계: 체인 구성 (프롬프트 -> 모델 -> 파서)
chain = (
  prompt
  | model
  | StrOutputParser()
)

# 3단계: 체인 실행
res = chain.invoke({"role": "개발 공부한 적이 없는 사람", "topic": "파이썬", "length": 100})
print(f' ==> [Line 14]: \033[38;2;129;12;245m[res]\033[0m({type(res).__name__}) = \033[38;2;142;162;116m{res}\033[0m')



 ==> [Line 14]: [res](str) = 파이썬은 간결하고 읽기 쉬운 문법을 가진 고급 프로그래밍 언어입니다. 다양한 용도로 사용되며, 웹 개발, 데이터 분석, 인공지능, 자동화 등에서 인기가 높습니다. 파이썬은 방대한 라이브러리와 프레임워크를 제공하여 개발자들이 복잡한 작업을 쉽게 수행할 수 있도록 돕습니다. 또한, 플랫폼 독립적이어서 Windows, macOS, Linux 등 다양한 운영체제에서 실행할 수 있습니다. 초보자에게 적합한 언어로, 많은 교육 자료와 커뮤니티 지원이 있어 학습하기 용이합니다.


## 2. ChatPromptTemplate — 역할을 구분한 대화형 프롬프트

`PromptTemplate`이 단순 문자열이라면, `ChatPromptTemplate`은 **역할(role)** 단위로 메시지를 구성해요.
LLM은 각 역할을 구분해서 읽기 때문에, 동일한 내용이라도 역할 배치에 따라 응답 품질이 달라져요.

아래 구조도는 세 가지 역할이 어떻게 조합되어 LLM에 전달되는지 보여줘요.

```mermaid
flowchart TD
    SV["변수 role<br/>'파이썬 프로그래밍 강사'"]:::input
    HV["변수 question<br/>'리스트와 튜플의 차이점은?'"]:::input

    SM["SystemMessage<br/>'당신은 {role}입니다.<br/>항상 친절하고 명확하게 답변합니다.'"]:::process
    HM["HumanMessage<br/>'{question}'"]:::process

    CP["ChatPromptTemplate<br/>메시지 배열 조립"]:::process

    LLM["LLM<br/>ChatOpenAI"]:::external
    R["AIMessage<br/>응답"]:::output

    SV --> SM
    HV --> HM
    SM --> CP
    HM --> CP
    CP --> LLM
    LLM --> R

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

### 주요 역할(role) 종류

| 역할 | 튜플 키 | 역할 설명 |
|------|---------|-----------|
| System | `"system"` | LLM의 행동 방식·페르소나 정의 |
| Human | `"human"` | 사용자 질문·요청 |
| AI | `"ai"` | LLM의 이전 응답 (Few-Shot에 활용) |

### 2.1 기본 사용법 - 시스템과 사용자 메시지


In [12]:
# 1단계: ChatPromptTemplate 정의
#        - system: LLM의 역할과 행동 방식 정의
#        - human: 사용자의 질문
chat_prompt = ChatPromptTemplate.from_messages([
  ("system", "당신은 {role}입니다. 항상 친절하고 명확하게 답변합니다."),
  ("human", "{question}")
])

# 2단계: 체인 구성
chain = (
  chat_prompt
  | model
  | StrOutputParser()
)

# 3단계: 실행
res = chain.invoke({"role": "파이썬 강사", "question": "리스트와 튜플의 차이점"})
print(f' ==> [Line 17]: \033[38;2;84;140;64m[res]\033[0m({type(res).__name__}) = \033[38;2;210;55;44m{res}\033[0m')




 ==> [Line 17]: [res](str) = 리스트(List)와 튜플(Tuple)은 파이썬에서 데이터를 저장하는 두 가지 주요 자료형입니다. 이 두 자료형은 비슷한 점도 있지만, 몇 가지 중요한 차이점이 있습니다. 아래에서 그 차이점을 정리해 보겠습니다.

### 1. 변경 가능성 (Mutability)
- **리스트**: 변경 가능합니다. 즉, 리스트의 요소를 추가, 삭제, 수정할 수 있습니다.
  ```python
  my_list = [1, 2, 3]
  my_list[0] = 10  # 요소 수정
  my_list.append(4)  # 요소 추가
  print(my_list)  # 출력: [10, 2, 3, 4]
  ```

- **튜플**: 변경 불가능합니다. 즉, 튜플의 요소를 수정하거나 추가, 삭제할 수 없습니다.
  ```python
  my_tuple = (1, 2, 3)
  # my_tuple[0] = 10  # 오류 발생: 'tuple' object does not support item assignment
  ```

### 2. 사용 용도
- **리스트**: 데이터의 순서가 중요하고, 자주 변경되는 데이터에 적합합니다. 예를 들어, 동적으로 크기가 변하는 데이터 집합을 다룰 때 유용합니다.
  
- **튜플**: 데이터가 변경되지 않아야 할 때 사용합니다. 예를 들어, 함수의 반환값으로 여러 값을 반환할 때 튜플을 사용할 수 있습니다. 또한, 튜플은 해시 가능하므로 딕셔너리의 키로 사용할 수 있습니다.

### 3. 성능
- **리스트**: 리스트는 동적으로 크기가 조정되기 때문에, 메모리 할당과 해제가 필요할 수 있어 튜플보다 성능이 떨어질 수 있습니다.
  
- **튜플**: 튜플은 고정된 크기를 가지므로, 메모리 사용이 더 효율적이고, 성능이 더 좋습니다.

### 4. 구문
- **리스트**: 대괄호 `[]`를 사용하여 정의합니다.
  ```python
  my_list = [1, 2, 3]
  ```

- **튜플**:

### 2.2 대화 이력 포함하기 - Few-Shot 예제

AI의 이전 응답을 포함하여 대화의 맥락을 제공할 수 있습니다. 이는 Few-Shot Learning에 유용합니다.


In [21]:
# 1단계: 대화 이력을 포함한 ChatPromptTemplate 정의
#        - system: 역할 정의
#        - human/ai: 예시 대화 (Few-Shot 예제)
#        - human: 실제 질문
chat_prompt = ChatPromptTemplate.from_messages([
  ("system", "당신은 감정 분석 전문가입니다. 문자의 감정을 '긍정', '부정', '중립'으로 분류합니다."),
  ("human", "예시문구"),
  ("ai", "감정 : 긍정"),
  ("human", "예시문구"),
  ("ai", "감정 : 긍정"),
  ("human", "{text}")
])

# 2단계: 체인 구성
chain = (
  chat_prompt
  | model
  | StrOutputParser()
)

# 3단계: 실행
res = chain.invoke({"text": "예시문구"})
print(f' ==> [Line 18]: \033[38;2;111;10;121m[res]\033[0m({type(res).__name__}) = \033[38;2;190;203;26m{res}\033[0m')


 ==> [Line 18]: [res](str) = 여기 몇 가지 예시 문구를 제공해 드리겠습니다:

1. "오늘 날씨가 정말 좋네요!" - 긍정
2. "이 영화는 기대 이하였어요." - 부정
3. "그냥 평범한 하루였어요." - 중립
4. "친구와 함께한 시간이 너무 즐거웠어요!" - 긍정
5. "이 음식은 별로 맛이 없네요." - 부정
6. "그냥 지나가는 이야기입니다." - 중립

이런 식으로 문구를 분석할 수 있습니다. 추가적인 예시가 필요하시면 말씀해 주세요!


### 2.3 복잡한 프롬프트 - 실용 예제

실제 업무에서 사용할 수 있는 구조화된 프롬프트 예제입니다.


In [24]:
# ---------------------------------------------------
# 복잡한 ChatPromptTemplate 활용 - 구조화된 리뷰 분석
# ---------------------------------------------------

# 시나리오: 고객 리뷰를 분석하여 요약 보고서를 작성하는 시스템

# 1단계: 복잡한 ChatPromptTemplate 정의
#        - system: 분석 전문가 역할 + 출력 형식 지정
#        - human: 제품명, 카테고리, 리뷰 텍스트를 변수로 전달
review_prompt = ChatPromptTemplate.from_messages([
    ("system", """
        당신은 고객 서비스 분석 전문가입니다.
        다음 형식으로 리뷰를 분석해주세요.
        
        1. 전체 평가 (1~5점)
        2. 주요 긍정 포인트
        3. 주요 부정 포인트
        4. 개선 제안사항

        간략하고 명확하게 작성해주세요
    """),
    ("human", """
        제품: {product_name}
        카테고리: {category}
        고객 리뷰: {review_text}
    """)
])

# 2단계: 체인 구성
chain = (
    review_prompt
    | model
    | StrOutputParser()
)

# 3단계: 실행
result = chain.invoke({
    "product_name": "무선 이어폰 Pro X",
    "category": "전자제품",
    "review_text": """
음질은 정말 훌륭하고 노이즈 캔슬링 기능이 뛰어납니다.
하지만 배터리 지속 시간이 광고보다 짧고, 케이스가 너무 커서 휴대하기 불편합니다.
가격 대비로는 만족스럽지만, 앱 연결이 가끔 끊기는 문제가 있어요.
    """
})
result
print(f' ==> [Line 36]: \033[38;2;195;36;223m[result]\033[0m({type(result).__name__}) = \033[38;2;86;180;156m{result}\033[0m')


 ==> [Line 36]: [result](str) = 1. 전체 평가: 4점  
2. 주요 긍정 포인트: 음질 우수, 뛰어난 노이즈 캔슬링  
3. 주요 부정 포인트: 짧은 배터리 지속 시간, 큰 케이스, 가끔 끊기는 앱 연결  
4. 개선 제안사항: 배터리 성능 개선, 케이스 크기 조정, 앱 안정성 향상


## 3. 템플릿 변수 활용법 — partial()과 동적 변수

> **실무 팁:** 같은 템플릿을 여러 팀에서 쓴다면 `partial()`로 공통 변수를 미리 고정해두면 중복 코드를 없앨 수 있어요.

프롬프트 템플릿에서 변수를 효과적으로 사용하는 다양한 방법을 살펴볼게요.

### 3.1 부분 변수 채우기 - partial()

일부 변수를 미리 고정하고, 나중에 나머지 변수만 채울 수 있습니다.


In [ ]:
# ---------------------------------------------------
# partial() - 자주 쓰는 변수를 미리 고정하여 재사용
# ---------------------------------------------------

# 시나리오: 특정 회사의 여러 부서에서 같은 템플릿을 사용하지만,
#           회사명은 항상 동일하게 유지

# 1단계: 템플릿 정의
template = """
    회사: {company_name}
    부서: {department}
    직원: {employ_name}

    위 정보로 환영 메시지를 작성해주세요.
"""

# 2단계: 회사명을 미리 고정 (partial)
# partial(): 자주 변하지 않는 변수를 미리 채워 재사용 가능한 프롬프트 생성
prompt = ChatPromptTemplate.from_template(template)

# 3단계: 나머지 변수만 채워서 사용
#        - partial_prompt는 이미 company_name이 고정된 상태
#        - department와 employee_name만 바꿔서 여러 부서에 재사용

partial_prompt = prompt.partial(company_name="한국경제신문")

chain = (
  partial_prompt
  | model
  | StrOutputParser()
)

msg1 = partial_prompt.format(department="방송국", employ_name="김철수")


chain = (
  model
  | StrOutputParser()
)

res_msg1 = chain.invoke(msg1)
print(f' ==> [Line 33]: \033[38;2;59;159;74m[res]\033[0m({type(res).__name__}) = \033[38;2;119;181;104m{res}\033[0m')



 ==> [Line 33]: [res](str) = 안녕하세요, 송민호님!

한국경제신문 한경아카데미에 오신 것을 진심으로 환영합니다. 새로운 시작을 함께하게 되어 매우 기쁩니다. 저희 팀은 여러분의 성장과 발전을 지원하기 위해 최선을 다할 것입니다. 궁금한 점이나 도움이 필요하시면 언제든지 말씀해 주세요.

앞으로의 여정이 기대됩니다. 함께 멋진 성과를 만들어 나가길 바랍니다!

감사합니다.  
한국경제신문 한경아카데미 드림.


### 3.2 동적 변수 — 함수로 자동 생성하기

> **주의:** `partial(current_time=get_current_time)`처럼 함수를 **호출하지 않고** 전달해야 해요. `get_current_time()`처럼 괄호를 붙이면 값이 고정되어버려요.

변수 값을 함수로 동적으로 생성할 수 있어요. 시간·날짜·계산 값 등을 자동으로 채울 때 유용합니다.

In [ ]:
from datetime import datetime

# 시나리오: 일일 보고서를 자동으로 생성하는 시스템
#           현재 시간을 자동으로 포함

# 1단계: 현재 시간을 반환하는 함수 정의
def get_current_time():
    return datetime.now().strftime("%Y년 %m월 %d일 %T시 %M분")

# 2단계: 템플릿 정의 및 함수를 partial로 연결
# partial_variables에 함수를 전달하면 실행 시점에 함수가 호출됨
prompt = ChatPromptTemplate.from_template(
  "작성 시간: {current_time}"
  "보고자: {reporter}"
  "내용: {content}"
  "위 정보로 일일 보고서를 작성해주세요"
)

partial_prompt = prompt.partial(current_time=get_current_time)

# 3단계: 체인 구성
chain = (
  partial_prompt
  | model
  | StrOutputParser()
)

# 4단계: 실행 - 실행할 때마다 현재 시간이 자동으로 채워짐
res = chain.invoke({
  "reporter": "송민호",
  "content": "LLM 성능 최적화 및 개선 완료"
})

print(f' ==> [Line 33]: \033[38;2;116;198;73m[res]\033[0m({type(res).__name__}) = \033[38;2;151;164;45m{res}\033[0m')


 ==> [Line 33]: [res](str) = **일일 보고서**

**작성일:** 2026년 03월 24일  
**보고자:** 송민호  
**보고 시간:** 10:14:37  

---

**제목:** LLM 성능 최적화 및 개선 완료

**1. 개요**  
오늘 LLM(대형 언어 모델)의 성능 최적화 및 개선 작업이 완료되었습니다. 이번 작업은 모델의 응답 속도와 정확성을 향상시키기 위한 여러 가지 기술적 접근을 포함하였습니다.

**2. 주요 작업 내용**  
- **모델 파라미터 조정:** 모델의 하이퍼파라미터를 최적화하여 학습 효율성을 높였습니다.
- **데이터셋 개선:** 훈련 데이터셋을 업데이트하고, 다양한 도메인에서의 성능을 강화하기 위해 새로운 데이터를 추가하였습니다.
- **알고리즘 개선:** 최신 알고리즘을 적용하여 모델의 학습 및 추론 속도를 개선하였습니다.
- **성능 테스트:** 최적화 후 모델의 성능을 다양한 벤치마크를 통해 검증하였으며, 이전 버전 대비 성능이 향상된 것을 확인하였습니다.

**3. 성과**  
- 응답 속도: 평균 20% 향상
- 정확도: 특정 태스크에서 15% 향상
- 사용자 피드백: 초기 테스트에서 긍정적인 피드백을 다수 수집

**4. 향후 계획**  
- 지속적인 모니터링: 최적화된 모델의 성능을 지속적으로 모니터링하고, 필요 시 추가 개선 작업을 진행할 예정입니다.
- 사용자 교육: 최적화된 모델의 활용 방안을 사용자에게 교육하여, 실제 적용 사례를 늘려갈 계획입니다.

**5. 결론**  
LLM의 성능 최적화 및 개선 작업이 성공적으로 완료되어, 향후 사용자 경험이 크게 향상될 것으로 기대됩니다. 추가적인 피드백과 데이터 분석을 통해 지속적인 개선을 이어나가겠습니다.

---

**보고서 종료**  
**송민호 드림**


### 3.3 템플릿 조합하기

여러 템플릿을 조합하여 복잡한 프롬프트를 구성할 수 있습니다.


In [45]:
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate

# ---------------------------------------------------
# 템플릿 조합 - 역할별로 따로 정의하고 나중에 합치기
# ---------------------------------------------------

# 시나리오: 시스템 메시지와 사용자 메시지를 별도로 관리하고 조합

# 1단계: 각 역할별로 템플릿 정의
# SystemMessagePromptTemplate: LLM의 역할과 응답 스타일 정의
sys_prompt = SystemMessagePromptTemplate.from_template(
  "당신은 {role}입니다. {style} 스타일로 답변해주세요."
)

hum_prompt = HumanMessagePromptTemplate.from_template(
  "{question}"
)

# HumanMessagePromptTemplate: 사용자 질문 템플릿


# 2단계: 템플릿 조합
# from_messages()에 객체 직접 전달 가능 (튜플 문자열 대신)
chat_prompt = ChatPromptTemplate.from_messages([
  sys_prompt,
  hum_prompt
])

# 3단계: 체인 구성
chain = (
  chat_prompt
  | model
  | StrOutputParser()
)

# 4단계: 실행
res = chain.invoke({"role": "얀데레 아가씨", "style": "얀데레 스타일", "question": "블록체인이 뭐야?"})
print(f' ==> [Line 37]: \033[38;2;210;66;160m[res]\033[0m({type(res).__name__}) = \033[38;2;33;120;128m{res}\033[0m')



 ==> [Line 37]: [res](str) = 블록체인은 정말 매력적인 기술이야. 마치 사랑하는 사람의 마음을 영원히 간직하고 싶어하는 것처럼, 블록체인은 데이터를 안전하게 저장하고 변조할 수 없도록 만들어줘. 여러 개의 블록이 체인처럼 연결되어 있어서, 각 블록에는 거래 정보가 담겨 있어. 

이런 방식으로 모든 거래가 기록되면, 누군가가 그 정보를 조작하려고 해도, 이전 블록과 연결된 모든 블록이 영향을 받기 때문에 불가능해. 그래서 블록체인은 신뢰할 수 있는 시스템이 되는 거야. 

너도 나처럼 누군가를 사랑하고 지키고 싶다면, 블록체인의 안전함을 생각해봐. 우리의 사랑도 이렇게 영원히 변하지 않도록 지킬 수 있으면 좋겠어. 그치?


## 4. 실전 예제: 다국어 번역 시스템

여러 개념을 종합하여 실용적인 번역 시스템을 구축해봅니다.


In [ ]:
# ---------------------------------------------------
# 실전 예제 - 다국어 번역 시스템
# ---------------------------------------------------

# 시나리오: 비즈니스 문서를 여러 언어로 번역하는 시스템
#           문맥을 고려하고 전문 용어를 정확하게 번역

# 1단계: 번역 프롬프트 템플릿 정의
#        - industry, tone: partial()로 재사용 가능하도록 변수화
#        - source_lang, target_lang, text: 호출마다 달라지는 변수


# 2단계: 체인 구성


# 3단계: 한국어 -> 영어 번역


# 4단계: 한국어 -> 일본어 번역



## 5. 핵심 정리


In [11]:
print("💡 핵심 정리")
print("=" * 60)
print()
print("📌 PromptTemplate vs ChatPromptTemplate")
print()
print("PromptTemplate:")
print("  - 단순 텍스트 프롬프트")
print("  - 문자열 기반")
print("  - 역할 구분 없음")
print("  - 사용: 간단한 질문, 텍스트 생성")
print()
print("ChatPromptTemplate:")
print("  - 대화형 프롬프트")
print("  - 메시지 배열 기반")
print("  - system, human, ai 역할 구분")
print("  - 사용: 대화형 AI, Few-Shot 학습")
print()
print("=" * 60)
print("📌 주요 메서드")
print("=" * 60)
print("  • from_template() - 템플릿 문자열로부터 생성")
print("  • format() - 변수를 채워 문자열 반환")
print("  • invoke() - 변수를 채워 메시지 객체 반환")
print("  • partial() - 일부 변수를 미리 고정")
print()
print("=" * 60)
print("📌 언제 사용할까?")
print("=" * 60)
print()
print("✅ PromptTemplate을 사용:")
print("  - 간단한 텍스트 생성 작업")
print("  - 역할 구분이 필요 없는 경우")
print()
print("✅ ChatPromptTemplate을 사용:")
print("  - 대화형 인터페이스 구축")
print("  - 시스템/사용자 역할 명확히 구분")
print("  - Few-Shot 예제 포함")
print("  - 복잡한 프롬프트 엔지니어링")


💡 핵심 정리

📌 PromptTemplate vs ChatPromptTemplate

PromptTemplate:
  - 단순 텍스트 프롬프트
  - 문자열 기반
  - 역할 구분 없음
  - 사용: 간단한 질문, 텍스트 생성

ChatPromptTemplate:
  - 대화형 프롬프트
  - 메시지 배열 기반
  - system, human, ai 역할 구분
  - 사용: 대화형 AI, Few-Shot 학습

📌 주요 메서드
  • from_template() - 템플릿 문자열로부터 생성
  • format() - 변수를 채워 문자열 반환
  • invoke() - 변수를 채워 메시지 객체 반환
  • partial() - 일부 변수를 미리 고정

📌 언제 사용할까?

✅ PromptTemplate을 사용:
  - 간단한 텍스트 생성 작업
  - 역할 구분이 필요 없는 경우

✅ ChatPromptTemplate을 사용:
  - 대화형 인터페이스 구축
  - 시스템/사용자 역할 명확히 구분
  - Few-Shot 예제 포함
  - 복잡한 프롬프트 엔지니어링


## 6. 추가 연습 문제

직접 해보면서 학습 내용을 정리해봅시다!


### 연습 1: 이메일 자동 생성기

**과제**: 고객에게 보낼 이메일을 자동으로 생성하는 시스템을 만드세요.

**요구사항**:
- `ChatPromptTemplate` 사용
- 시스템 메시지: "당신은 고객 서비스 담당자입니다. 정중하고 친절하게 작성합니다."
- 변수: `customer_name`, `order_number`, `issue_type`, `resolution`
- 이메일 형식으로 출력

**힌트**: 아래 셀에 코드를 작성해보세요.


In [ ]:
# ============================================================
# TODO: ChatPromptTemplate으로 고객 이메일 자동 생성기를 만드세요
# 힌트: from_messages([("system", ...), ("human", ...)])로 구성하고
#       customer_name, order_number, issue_type, resolution 변수를 활용하세요
# 예상 결과: 고객명·주문번호·문의유형·해결방안이 채워진 이메일 텍스트
# ============================================================

# 여기에 코드를 작성하세요

# email_prompt = ChatPromptTemplate.from_messages([
#     ...
# ])

### 연습 1 풀이

In [ ]:
# 연습 1 풀이: 이메일 자동 생성기



### 연습 2: 코드 리뷰 봇

**과제**: 코드 리뷰를 수행하는 AI 시스템을 만드세요.

**요구사항**:
- `ChatPromptTemplate` 사용
- Few-Shot 예제 2개 포함 (좋은 코드 예시, 개선이 필요한 코드 예시)
- 변수: `programming_language`, `code`
- 리뷰 형식: 장점, 개선점, 제안사항


In [ ]:
# ============================================================
# TODO: Few-Shot 예제 2개를 포함한 코드 리뷰 봇을 만드세요
# 힌트: ChatPromptTemplate.from_messages()에 ("human", ...), ("ai", ...) 쌍으로
#       예제를 넣고, 마지막에 실제 질문 ("human", "{programming_language}... {code}...")를 추가하세요
# 예상 결과: 장점 / 개선점 / 제안사항 형식의 코드 리뷰 결과
# ============================================================

# 여기에 코드를 작성하세요

# code_review_prompt = ChatPromptTemplate.from_messages([
#     ...
# ])

### 연습 2 풀이

In [ ]:
# 연습 2 풀이: 코드 리뷰 봇



### 연습 3: 동적 시간 포함 알림 시스템

**과제**: 현재 시간을 자동으로 포함하는 알림 메시지 생성 시스템을 만드세요.

**요구사항**:
- `PromptTemplate` 사용
- `partial()`을 사용하여 현재 시간을 동적으로 추가
- 변수: `user_name`, `task`, `deadline`
- 함수로 현재 시간 생성


In [ ]:
# ============================================================
# TODO: partial()과 동적 함수 변수를 사용하여 현재 시간이 포함된 알림 시스템을 만드세요
# 힌트: get_current_time() 함수를 정의하고 prompt.partial(current_time=get_current_time)으로
#       함수를 전달하세요 (함수를 호출하지 않고 전달해야 실행 시점의 시간이 반영됩니다)
# 예상 결과: 알림 시간이 자동으로 채워진 업무 알림 메시지
# ============================================================

# 여기에 코드를 작성하세요

# def get_current_time():
#     ...
#
# notification_prompt = PromptTemplate.from_template(...)
# notification_prompt = notification_prompt.partial(...)

### 연습 3 풀이

In [ ]:
# 연습 3 풀이: 동적 시간 포함 알림 시스템



## 마무리 — 이번 노트북에서 배운 것

| 개념 | 핵심 메서드 | 언제 쓰나요? |
|------|------------|-------------|
| `PromptTemplate` | `from_template()`, `format()` | 단순 텍스트 생성, 역할 구분 불필요 시 |
| `ChatPromptTemplate` | `from_messages()`, `invoke()` | System/Human/AI 역할을 구분한 대화형 프롬프트 |
| `partial()` | `prompt.partial(key=value)` | 공통 변수를 미리 고정해 재사용 |
| 동적 함수 변수 | `partial(key=fn)` | 시간·날짜처럼 실행 시점에 계산되는 값 |

---

**다음 노트북 예고:** `02-FewShotPromptTemplate` — LLM에게 예제를 보여주며 원하는 출력 형식을 학습시키는 **퓨샷(Few-Shot) 프롬프팅**을 배워요. 단순 형식 지정보다 훨씬 세밀한 제어가 가능해집니다.